<a href="https://colab.research.google.com/github/VioletteGL/Interpretabiidad-y-Causalidad-Bourbaki/blob/main/I%26C_2026_Recomendacion.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Sistemas de recomendación y causalidad. Estimación del sesgo.

Los sistemas de recomendación son algoritmos diseñados para sugerir elementos relevantes a los usuarios. Su objetivo principal es predecir la “calificación” o “preferencia” que un usuario daría a un item.

Existen tres enfoques principales:
1. **Filtrado Colaborativo (CF):** se apoya en similitudes entre usuarios o ítems.
2. **Basado en Contenido:** usa características de los ítems para recomendar.
3. **Modelos Híbridos:** combinan ambos.


In [6]:
import warnings
warnings.filterwarnings("ignore")

In [7]:
# Temporarily uninstall scikit-surprise and downgrade numpy
!pip uninstall -y scikit-surprise implicit numpy
!pip install numpy==1.26.4 scikit-surprise implicit

# IMPORTANT: After running this cell, you MUST restart the Colab runtime
# (Runtime -> Restart runtime) and then re-run all cells from the beginning.

Found existing installation: scikit-surprise 1.1.4
Uninstalling scikit-surprise-1.1.4:
  Successfully uninstalled scikit-surprise-1.1.4
Found existing installation: implicit 0.7.2
Uninstalling implicit-0.7.2:
  Successfully uninstalled implicit-0.7.2
Found existing installation: numpy 1.26.4
Uninstalling numpy-1.26.4:
  Successfully uninstalled numpy-1.26.4
  Using cached numpy-1.26.4-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (61 kB)
  Using cached scikit_surprise-1.1.4-cp312-cp312-linux_x86_64.whl
  Using cached implicit-0.7.2-cp312-cp312-linux_x86_64.whl
Using cached numpy-1.26.4-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (18.0 MB)
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
rasterio 1.5.0 requires numpy>=2, but you have numpy 1.26.4 which is incompatible.
jax 0.7.2 requires numpy>=2.0, but you have numpy 1.26.4 w

In [8]:
#!pip install implicit

* Otra dependencia para factorización de matrices es LightFM

### 1. Filtrado Colaborativo (CF)

Este es el enfoque más común, realiza predicciones basadas en los comportamientos pasados de un gran grupo de usuarios.

Hay dos subtipos principales:

-   **Filtrado colaborativo basado en usuarios:** encuentra usuarios similares al usuario objetivo (es decir, que han calificado ítems de manera parecida en el pasado) y recomienda ítems que estos usuarios similares han disfrutado pero que el usuario objetivo aún no ha visto.

*Analogía:* es como preguntar a tus amigos con gustos similares en películas que te recomienden una.

-  **Filtrado colaborativo basado en items:** en lugar de buscar usuarios similares, encuentra ítems similares a los que el usuario ya ha calificado positivamente. Calcula la similitud en función de quién los ha visto, gustado o comprado. Ejemplo: usuarios que compraron Álgebra Lineal y sus Aplicaciones también compraron con frecuencia Introducción a los Algoritmos.

 Es como la función de Amazon “Clientes que compraron este producto también compraron...”.

**Ventajas:**

- No requiere conocer las características de los ítems ni de los usuarios -(agnóstico al contenido).

- Puede generar recomendaciones inesperadas (serendipia), sugiriendo ítems que el usuario no habría descubierto por sí mismo.

**Desventajas:**

- Problema de arranque en frío: difícil recomendar a nuevos usuarios o evaluar ítems nuevos sin historial.

- Escasez de datos: el rendimiento baja cuando la matriz de interacciones usuario-ítem es muy dispersa (sparse).

#### Factorización Matricial (SVD)
Una técnica muy usada en CF es la inspirada en la **factorización matricial (Singular Value Descomposition)**:

Descompone la gran matriz dispersa de interacciones usuario-item (R) en dos matrices más pequeñas y densas:

- Una de características de usuarios (P)  
- Otra de características de items (Q)  

El producto punto entre un vector de usuario en (P) y uno de ítem en (Q) aproxima la calificación esperada:

$$
R \approx P\, Q^{\top}
$$

Con esto podemos predecir la preferencia de un usuario por un item.

Pero si los datos de \(R\) están sesgados (ej. solo vemos películas populares), entonces la factorización **también aprenderá el sesgo**.

## El Problema de los Confusores
- **Confusores (confounders):** Son variables que afectan tanto la exposición como el resultado.
- Ejemplos:
  - Popularidad: lo más visto parece “mejor”.
  - Posición en pantalla: lo que aparece primero recibe más clics.
  - Contexto: día, dispositivo, campañas, etc.

### Consecuencias:
- **Sesgo hacia lo popular.**
- **Feedback loop:** lo popular se recomienda más → genera más datos → se recomienda más.  
- **Evaluación sesgada:** las métricas tradicionales pueden sobreestimar el rendimiento.

Por lo tanto necesitamos técnicas que **corrijan este sesgo**.

## Ejemplo
Para abordar el problema anterior realizaremos un sistema de recomendacion con la base movielens la cual contiene información de calificaciones y etiquetas de películas recolectadas en la plataforma [MovieLens](http://movielens.org), un sistema de recomendación de películas desarrollado por **GroupLens Research** de la Universidad de Minnesota.

- **Tamaño del dataset:**  
  - **100,836** calificaciones  
  - **3,683** etiquetas (tags)  
  - **9,742** películas  
  - **610** usuarios  

- **Periodo de recolección:** Desde **29 de marzo de 1996** hasta **24 de septiembre de 2018**.  
- **Última generación del dataset:** **26 de septiembre de 2018**.  
- **Condición de selección de usuarios:** cada usuario incluido ha calificado al menos **20 películas**.  
- **Nota:** No incluye datos demográficos de los usuarios (solo un `userId` anónimo).

### Archivos del Dataset
El dataset está compuesto por cuatro archivos CSV principales:

#### 1. `ratings.csv`
Contiene todas las calificaciones hechas por los usuarios.
- **Variables:**  userId, movieId, rating, timestamp

- **Detalles:**  
 - `rating` va de **0.5 a 5.0** en incrementos de 0.5 estrellas.  
 -  `timestamp` representa segundos desde el 1 de enero de 1970 (UTC).

#### 2. `tags.csv`
Contiene las etiquetas (palabras clave o frases cortas) aplicadas por usuarios a las películas.
- **Variables:** userId, movieId, tag, timestamp

- **Detalles:**  
 - `tag` son metadatos generados por usuarios.  
 - `timestamp` en el mismo formato que en `ratings.csv`.

#### 3. `movies.csv`
Contiene la información básica de cada película.
- **Variables:** movieId, title, genres

- **Detalles:**  
 - `title`: nombre de la película con el año en paréntesis (ej. *Toy Story (1995)*).  
 - `genres`: lista separada por barras verticales `|` (ej. *Adventure|Animation|Children|Comedy|Fantasy*).  
 - Posibles géneros incluyen: *Action, Adventure, Animation, Comedy, Drama, Fantasy, Horror, Romance, Sci-Fi, Thriller, War, Western,* entre otros.

#### 4. `links.csv`
Contiene identificadores que permiten enlazar las películas con otras bases de datos.
- **Variables:**  movieId,imdbId,tmdbId
- **Detalles:**  
 - `imdbId`: corresponde al identificador de [IMDb](http://www.imdb.com).  
 - `tmdbId`: corresponde al identificador de [The Movie Database (TMDb)](https://www.themoviedb.org).  


 ### Baseline sin corrección (SVD simple)

In [2]:
import pandas as pd

# Cargamos ratings y movies
ratings = pd.read_csv( "/content/sample_data/ratings.csv")
movies = pd.read_csv("/content/sample_data/movies.csv")

ratings.head()

,userId,movieId,rating,timestamp
0,1,1,4.0,964982703
1,1,3,4.0,964981247
2,1,6,4.0,964982224
3,1,47,5.0,964983815
4,1,50,5.0,964982931


In [3]:
movies.head()

,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy


In [4]:
movies.shape

(9742, 3)

In [5]:
ratings.rating.value_counts()
### Observemos que no hay calificación de cero

,count
rating,
4.0,26818
3.0,20047
5.0,13211
3.5,13136
4.5,8551
2.0,7551
2.5,5550
1.0,2811
1.5,1791


### El problema del sesgo en la factorización

La **factorización matricial (SVD)** nos permite aproximar la matriz de interacciones:

$$
R \approx P Q^T
$$

donde:
- $P$ representa características latentes de usuarios.  
- $Q$ representa características latentes de ítems.  

Con esto podemos predecir la preferencia de un usuario por un ítem.

**Pero hay un problema importante:** si los datos de $R$ están sesgados (ejemplo: solo vemos películas populares porque fueron más expuestas), entonces **la factorización también aprenderá ese sesgo**.  

En la práctica, esto significa que el modelo recomendará **más de lo mismo** (lo popular), reforzando el loop de retroalimentación.


### Para corregir el sesgo: IPS

La clave está en reconocer que **no todos los pares usuario–item tienen la misma probabilidad de aparecer en los datos**.  
A esto lo llamamos **propensión**:

$$
e_{ui} = \Pr(A_{ui} = 1 \mid u,i)
$$
(variable P de la base de datos para el modelo causal)
Donde:

- $u$: índice del **usuario** (ej. Juan, usuario #15).  
- $i$: índice del **ítem** (ej. Película X, ítem #203).
- $A_{ui}=1$ significa que el usuario $u$ tuvo la oportunidad de ver el ítem $i$.
- $e_{ui}$: la **propensión**, es decir, la probabilidad de que el par usuario–item $(u,i)$ se **muestre** en los datos.  
  - Ejemplo: si una película aparece en la página principal y la ven el 80% de los usuarios, su $e_{ui}$ será alto.  
  - Si una película rara apenas aparece, su $e_{ui}$ será bajo.


#### Inverse Propensity Scoring (IPS)
La idea es **reponderar cada observación** de acuerdo a su propensión.  
Así, los pares muy expuestos (ej. ítems populares) pesan menos, y los pares poco expuestos pesan más.

$$
w_{ui} = \frac{1}{e_{ui}}
$$

- Si $e_{ui}$ es **alto** (ítem popular, siempre mostrado) → $w_{ui}$ es pequeño.  
- Si $e_{ui}$ es **bajo** (ítem de nicho, poco mostrado) → $w_{ui}$ es grande.  

Esto balancea el entrenamiento y reduce el sesgo de popularidad.


### Ejemplo IPS

Imaginemos dos películas:

- **Película A (blockbuster):** fue mostrada al **90%** de los usuarios.  
  - Propensión: $e_{ui} = 0.9$
  - Peso IPS:  
  $$
  w_{ui} = \frac{1}{0.9} \approx 1.11
  $$

- **Película B (de nicho):** fue mostrada solo al **10%** de los usuarios.  
  - Propensión: $e_{ui} = 0.1$
  - Peso IPS:  
  $$
  w_{ui} = \frac{1}{0.1} = 10
  $$


### Interpretación

- Si un error ocurre con **Película A** (muy popular), su impacto en el entrenamiento es bajo $(w \approx 1)$.  
- Si un error ocurre con **Película B** (poco expuesta), su impacto es **10 veces mayor**.

Esto significa que el modelo aprende a **no dejar de lado los ítems poco expuestos**, corrigiendo el sesgo hacia lo popular.

### La corrección con IPS
En IPS, no reemplazamos \(R\), sino que ajustamos **la importancia de cada entrada** en la factorización.  

$$
\tilde{R}_{ui} = w_{ui} \cdot R_{ui} \quad \text{con} \quad w_{ui} = \frac{1}{e_{ui}}
$$

- $\tilde{R}$ = versión **reponderada** de la matriz de interacciones.  

### Factorización reponderada

En lugar de minimizar solo el error sobre $R$, resolvemos:

$$
\min_{P,Q} \sum_{(u,i)\in \Omega} w_{ui} \, \big( R_{ui} - P_u \cdot Q_i \big)^2
$$

Donde:
- $\Omega$ es el conjunto de pares usuario–ítem observados.  
- Cada error se multiplica por $w_{ui}$.  


### Implementación

En nuestro caso:
- Usaremos una **regresión logística** para la propensión.
- Calculamos los pesos $w_{ui}$ (con clipping para evitar valores extremos).  
- Entrenamos un modelo de factorización.  

En librerías como **LightFM o implicit**, esto se implementa fácilmente pasando los $w_{ui}$ como `sample_weight` en el entrenamiento.

El resultado sera un modelo que no solo busca **minimizar el error**, sino que además **corrige el sesgo de exposición**, generando recomendaciones más justas y diversas.

In [6]:
movies[movies['movieId']==356]

,movieId,title,genres
314,356,Forrest Gump (1994),Comedy|Drama|Romance|War


### Comparación: Baseline vs IPS
Veremos:
1. RMSE (aprox. igual o un poco peor con IPS, pero más **justo**).
2. **Popularidad promedio** de ítems recomendados (con IPS podría bajar).

In [7]:
import pandas as pd
from sklearn.linear_model import LogisticRegression

from surprise import Dataset, Reader, SVD
from sklearn.metrics import root_mean_squared_error
import implicit
from scipy import sparse
import numpy as np
import random

# ── SPLIT ÚNICO ──
# Se ordena por usuario y timestamp para usar la última interacción de cada
# usuario como dato de prueba (evaluación temporal realista)

r_sorted = ratings.sort_values(["userId","timestamp"])
last_idx = r_sorted.groupby("userId").tail(1).index
test_df = r_sorted.loc[last_idx]
train_df = r_sorted.drop(index=last_idx)

pop = train_df["movieId"].value_counts().to_dict()

# ── BASELINE SVD (surprise) ──────
# SVD factoriza la matriz de ratings en factores latentes de usuarios e ítems.
# Se usa como baseline sin corrección de sesgo de exposición.

reader = Reader(rating_scale=(0.5, 5.0))
data_surprise = Dataset.load_from_df(train_df[["userId","movieId","rating"]], reader)
trainset = data_surprise.build_full_trainset()  ### estructura optimizada para Surprise

algo = SVD(n_factors=50, n_epochs=20, random_state=42)
algo.fit(trainset)

# Evaluación del baseline: se recogen ratings reales y predichos para calcular RMSE
y_true_svd, y_pred_svd = [], []
for r in test_df.itertuples(index=False):
    y_true_svd.append(float(r.rating))
    y_pred_svd.append(algo.predict(r.userId, r.movieId).est)  #### rating estimado por SVD

rmse_svd = root_mean_squared_error(y_true_svd, y_pred_svd)
print("RMSE Baseline (SVD sin IPS):", rmse_svd)

def recs_svd(uid, n=10):
    """Genera top-n recomendaciones para un usuario usando SVD baseline."""
    all_items = ratings["movieId"].unique()
    seen = set(train_df[train_df["userId"] == uid]["movieId"])
    preds = [(iid, algo.predict(uid, iid).est) for iid in all_items if iid not in seen]
    preds.sort(key=lambda x: -x[1])
    return [iid for iid, _ in preds[:n]]

# ── REINDEXADO (para implicit) ──
# implicit requiere índices contiguos 0...N-1 para usuarios e ítems

uids = {u:i for i,u in enumerate(sorted(ratings["userId"].unique()))}
iids = {m:i for i,m in enumerate(sorted(ratings["movieId"].unique()))}
U, I = len(uids), len(iids)
idx_to_mid = {v: k for k, v in iids.items()}

# ── PROPENSIÓN (modelo logístico) ─
# Se estima la probabilidad de exposición e(u,i) = P(observado | usuario u, ítem i)
# usando actividad del usuario y popularidad del ítem como features
ua = train_df["userId"].value_counts().to_dict()
ip = train_df["movieId"].value_counts().to_dict()

# Construcción del dataset de exposición:
# positivos (E=1): pares observados en train
# negativos (E=0): pares no observados muestreados aleatoriamente
observed = set(zip(train_df["userId"], train_df["movieId"]))
rows = []

# Positivos: todas las interacciones en train
for r in train_df.itertuples(index=False):
    rows.append([ua[r.userId], ip[r.movieId], 1])


# Negativos: pares usuario-ítem no observados (mismo número que positivos → dataset balanceado)
users = ratings["userId"].unique()
items = ratings["movieId"].unique()
for _ in range(len(train_df)):
    u = random.choice(users)
    i = random.choice(items)
    if (u, i) not in observed:
        rows.append([ua.get(u,0), ip.get(i,0), 0])

exp_df = pd.DataFrame(rows, columns=["ua","ip","exposed"])

# Entrenamiento del modelo logístico de propensión
X = exp_df[["ua","ip"]] ### features: actividad del usuario y popularidad del ítem
y = exp_df["exposed"]   ### target: 1 si fue observado, 0 si no
logit = LogisticRegression(max_iter=500)
logit.fit(X, y)

def e_ui(u, i):
    """Estima la propensión de exposición P(observado | u, i) con el modelo logístico."""
    return logit.predict_proba([[ua.get(u,0), ip.get(i,0)]])[0,1]

# ── MATRIZ CON PESOS IPS
# Se construyen dos matrices sparse en formato item×user (requerido por implicit):
# R: matriz de ratings observados
# W: matriz de pesos IPS = 1/e(u,i) → penaliza ítems populares, premia los poco vistos

R = sparse.lil_matrix((I, U), dtype=np.float32)
W = sparse.lil_matrix((I, U), dtype=np.float32)

for r in train_df.itertuples(index=False):
    u_idx = uids[r.userId]
    i_idx = iids[r.movieId]
    weight = 1 / max(e_ui(r.userId, r.movieId), 1e-6) ### para evitar división por cero
    R[i_idx, u_idx] = float(r.rating)
    W[i_idx, u_idx] = weight

R_csr = R.tocsr()
W_csr = W.tocsr()

# ── ENTRENAMIENTO implicit (ALS con pesos IPS)
# ALS factoriza la matriz de pesos IPS alternando la optimización de factores
# de usuarios e ítems. Al usar W en lugar de R, el modelo aprende a desesgar
# las recomendaciones hacia ítems poco populares.

model = implicit.als.AlternatingLeastSquares(
    factors=32,       # dimensión del espacio latente
    iterations=15,    # número de alternaciones usuario-ítem
    random_state=42
)
model.fit(W_csr.T)  # implicit espera user × item

# ── PREDICCIONES Y RMSE ───────
# Nota: ALS no predice ratings directamente, sino scores de ranking.
# El RMSE aquí no es comparable con el del SVD baseline (escalas distintas).
user_factors = model.user_factors
item_factors = model.item_factors

y_true_lfm, y_pred_lfm = [], []
for r in test_df.itertuples(index=False):
    u_idx = uids[r.userId]
    i_idx = iids[r.movieId]
    score = float(np.dot(user_factors[u_idx], item_factors[i_idx]))
    y_true_lfm.append(float(r.rating))
    y_pred_lfm.append(score)

rmse_lfm = root_mean_squared_error(y_true_lfm, y_pred_lfm)
print("RMSE implicit ALS + IPS:", rmse_lfm)

# ── FUNCIÓN DE RECOMENDACIONES ─
def recommend_implicit(uid_raw, n=10):
    uid = uids[uid_raw]
    user_items = R_csr.T.tocsr()  ## formato user×item requerido por model.recommend()
    user_row = user_items[uid]
    item_indices, scores = model.recommend(
        uid, user_row, N=n + 200, filter_already_liked_items=True   # filtra ítems ya vistos
    )
    seen = set(train_df[train_df["userId"] == uid_raw]["movieId"])
    recs = []
    for idx, score in zip(item_indices, scores):
        mid = idx_to_mid[idx]
        if mid not in seen:
            recs.append((mid, score))
    recs.sort(key=lambda x: -x[1])  # ordena de mayor a menor
    return [mid for mid, _ in recs[:n]]

# ── POPULARIDAD MEDIA ──
# Mide si las recomendaciones se concentran en ítems populares o diversifican.
# Un valor menor indica que el modelo recomienda más ítems de nicho (menos sesgo).
def avg_pop(method_fn, users_eval, n=1000):
    vals = []
    for u in users_eval:
        recs = method_fn(u, n=n)
        vals.append(np.mean([pop.get(m, 0) for m in recs]))
    return np.mean(vals)

# Se evalúa sobre los primeros 500 usuarios del conjunto de prueba
users_eval = list(test_df["userId"].unique())[:500]

print("Popularidad media Top-1000 (Baseline SVD):", avg_pop(recs_svd, users_eval, n=1000))
print("Popularidad media Top-1000 (implicit ALS + IPS):", avg_pop(recommend_implicit, users_eval, n=1000))

RMSE Baseline (SVD sin IPS): 0.9681918149329729


Streaming output truncated to the last 5000 lines.
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LogisticRegression was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LogisticRegression was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LogisticRegression was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LogisticRegression was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LogisticRegression was fitted with feature na

  0%|          | 0/15 [00:00<?, ?it/s]

RMSE implicit ALS + IPS: 3.674542324277582
Popularidad media Top-1000 (Baseline SVD): 31.251744
Popularidad media Top-1000 (implicit ALS + IPS): 30.381802


Output:


```
RMSE implicit ALS + IPS: 3.6742683161909637
Popularidad media Top-1000 (Baseline SVD): 31.251744
Popularidad media Top-1000 (implicit ALS + IPS): 30.38153```



Coverage mide qué fracción del catálogo total de películas aparece en las recomendaciones generadas para los 500 usuarios evaluados con listas de top-1000.

SVD cubre el 68.3% del catálogo: hay un 31.7% de películas que nunca recomienda a nadie, concentrándose en un subconjunto más reducido de ítems.
ALS + IPS cubre el 75.0% del catálogo: logra recomendar a una porción mayor del catálogo, dejando fuera solo el 25%

¿Por qué ALS+IPS tiene mayor coverage?
Precisamente por los pesos IPS. Al asignar mayor peso a interacciones con ítems poco populares (que tienen menor propensión de ser vistos), el modelo aprende mejores representaciones de ítems de nicho, y por tanto los incluye más en sus recomendaciones.
SVD en cambio no corrige este sesgo, por lo que tiende a concentrar sus recomendaciones en ítems populares que tienen más señal en los datos de entrenamiento.

In [8]:
def coverage(method_fn, users_eval, n=1000):
    rec_items = set()
    for u in users_eval:
        recs = method_fn(u, n=n)
        rec_items.update(recs)
    total_items = len(ratings["movieId"].unique())
    return len(rec_items) / total_items

print("Coverage Top-1000 (Baseline SVD):", coverage(recs_svd, users_eval, n=1000))
print("Coverage Top-1000 (implicit ALS + IPS):", coverage(recommend_implicit, users_eval, n=1000))

Coverage Top-1000 (Baseline SVD): 0.6826408885232415
Coverage Top-1000 (implicit ALS + IPS): 0.7514397367338543


Output:



```
Coverage Top-1000 (Baseline SVD): 0.6826408885232415
Coverage Top-1000 (implicit ALS + IPS): 0.7514397367338543
```



# Conclusiones
- Los **confusores** sesgan la exposición y refuerzan lo popular.
- El **baseline (SVD sin corrección)** recomienda ítems más populares.
- Con **IPS (implicit con sample_weight)** equilibramos el entrenamiento:
  - Similar RMSE, pero recomendaciones menos sesgadas hacia la popularidad.
  - Mayor diversidad en el ranking.
- En producción: modelar propensión con más señales (posición, campaña, dispositivo).


# Usemos el recomendador

Vamos a generar una recomendación para todos los usuarios utilizando la recomendación de surprise. Generaremos 100 recomendaciones para cada uno.

Etiquetaremos a cada película con su exposición como sigue:

- E = 1 si el ítem aparece en el top-N recomendado por SVD para ese usuario
- E = 0 si no aparece

Este proceso va a tardar un poco

In [9]:
import pandas as pd
import numpy as np
from tqdm import tqdm

# ── PASO 1: generar recomendaciones con SVD baseline ──────────────────────────
# E = 1 si el ítem aparece en el top-N recomendado por SVD para ese usuario
# E = 0 si no aparece

N = 100  # tamaño de la lista de recomendación

exposure_rows = []
all_users = ratings["userId"].unique()
all_items = ratings["movieId"].unique()

for uid_raw in tqdm(all_users, desc="Generando exposiciones SVD"):
    recs_set = set(recs_svd(uid_raw, n=N))  # top-N del SVD baseline
    for mid in all_items:
        exposure_rows.append({
            "userId":  uid_raw,
            "movieId": mid,
            "E_svd":   1 if mid in recs_set else 0
        })

exposure_df = pd.DataFrame(exposure_rows)
print("Shape exposure_df:", exposure_df.shape)
print(exposure_df["E_svd"].value_counts())

Generando exposiciones SVD: 100%|██████████| 610/610 [00:59<00:00, 10.32it/s]


Shape exposure_df: (5931640, 3)
E_svd
0    5870640
1      61000
Name: count, dtype: int64


In [10]:
exposure_df

,userId,movieId,E_svd
0,1,1,0
1,1,3,0
2,1,6,0
3,1,47,0
4,1,50,0
...,...,...,...
5931635,610,160341,0
5931636,610,160527,0
5931637,610,160836,0
5931638,610,163937,0


In [11]:
from google.colab import files

# Guardar el CSV
exposure_df.to_csv("exposure_svd.csv", index=False)

# Descargar a tu computadora para usar en el reto del módulo V
files.download("exposure_svd.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>